# Phase 1 — FOV Correction & Layer Assignment (DREADD saline/DCZ cohort)

Goal: to capture curved/unevenly tilted L4 band correctly instead of a flat cutoff

**Landmark clicked:** L4 band center (same convention as the existing
`pick_layer4_peak`) — L2/3, L5, L6 are derived from it via the existing fixed
±70µm / 150µm anatomical offsets, now applied relative to the fitted curve
instead of a constant.

**Curve fit:** linear (`degree=1`) by default.

**Reference image for clicking:** suite2p `max_proj`, padded to the full
`Ly×Lx` frame (see Function 1) so it aligns with `med_coords` pixel coordinates.

In [34]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import json
import numpy as np
import matplotlib
matplotlib.use('Qt5Agg')  # interactive ginput() clicking + popup windows
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# One real DREADD session to develop and sanity-check each function against
# before we touch anything else. Swap this as needed while building.
# TEST_SESSION_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260722_JSY_JSY090_LongitudinalImaging_DREADD_Day4\TSeries-07222026-1831-002"
TEST_SESSION_DIR = r'D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260723_JSY_JSY093_LongitudinalImaging_DREADD_Day5\TSeries-07232026-0843-001'
TEST_PLANE0 = os.path.join(TEST_SESSION_DIR, 'suite2p', 'plane0')
UM_PER_PIXEL = 1.08952017715202  # V1_prism_DREADD animals (JSY090, JSY093)


## Function 1.1 — `get_fullframe_max_proj`

suite2p's `ops['max_proj']` is cropped to `ops['yrange']`/`ops['xrange']`
(the registered-movie valid region), so it's **smaller** than the full
`(Ly, Lx)` frame that `med_coords` (from `stat[i]['med']`), `mean_img`, and
every existing layer-assignment function assume.

Checked this on `TEST_SESSION_DIR`'s `ops.npy`: `Ly,Lx = 760,760` but
`max_proj.shape = (732, 748)`, with `yrange=[14,746]`, `xrange=[6,754]` —
exactly `746-14=732` and `754-6=748`. If we plotted `max_proj` directly and
overlaid `med_coords` on it, every ROI would be offset from where it actually
sits in the image.

**Fix:** embed `max_proj` into a zero-padded `(Ly, Lx)` canvas at that same
offset, so it drops into all downstream clicking/plotting code exactly like
`mean_img` does — no coordinate shifting needed anywhere else.

- **Input:** `ops` (dict from suite2p's `ops.npy`) — needs `Ly`, `Lx`,
  `max_proj`, `yrange`, `xrange`.
- **Output:** `(Ly, Lx)` float array, `max_proj` values inside the valid
  region, zero elsewhere.

In [35]:
def get_fullframe_max_proj(ops):
    """
    Embed suite2p's cropped ops['max_proj'] into a full (Ly, Lx) canvas at
    its registered yrange/xrange offset, so it aligns pixel-for-pixel with
    med_coords and every other full-frame image (mean_img, etc.) used
    elsewhere in this pipeline.

    Parameters
    ----------
    ops : dict
        suite2p ops dictionary (ops.npy). Needs 'Ly', 'Lx', 'max_proj',
        'yrange', 'xrange'.

    Returns
    -------
    fullframe : numpy.ndarray, shape (Ly, Lx)
        max_proj values inside [yrange[0]:yrange[1], xrange[0]:xrange[1]],
        zero-padded outside.
    """
    Ly, Lx = ops['Ly'], ops['Lx']
    y0, y1 = ops['yrange']
    x0, x1 = ops['xrange']
    max_proj = ops['max_proj']

    expected_shape = (y1 - y0, x1 - x0)
    if max_proj.shape != expected_shape:
        raise ValueError(
            f"max_proj shape {max_proj.shape} doesn't match yrange/xrange "
            f"{expected_shape} — check this session's ops.npy."
        )

    fullframe = np.zeros((Ly, Lx), dtype=max_proj.dtype)
    fullframe[y0:y1, x0:x1] = max_proj
    return fullframe


In [36]:
# --- Sanity check on the real test session ---
ops_test = np.load(os.path.join(TEST_PLANE0, 'ops.npy'), allow_pickle=True).item()
stat_test = np.load(os.path.join(TEST_PLANE0, 'stat.npy'), allow_pickle=True)
iscell_test = np.load(os.path.join(TEST_PLANE0, 'iscell.npy'), allow_pickle=True)

cell_idx_test = np.where(iscell_test[:, 0] == 1)[0]
med_coords_test = np.array([stat_test[i]['med'] for i in cell_idx_test])  # (n_cells, 2) -> (y, x)

fullframe_maxproj = get_fullframe_max_proj(ops_test)
print(f"Ly, Lx = {ops_test['Ly']}, {ops_test['Lx']}")
print(f"raw max_proj shape: {ops_test['max_proj'].shape}")
print(f"padded fullframe shape: {fullframe_maxproj.shape}")
print(f"n_cells: {len(med_coords_test)}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(fullframe_maxproj, cmap='gray',
          vmin=np.percentile(fullframe_maxproj, 2), vmax=np.percentile(fullframe_maxproj, 98))
ax.scatter(med_coords_test[:, 1], med_coords_test[:, 0], s=8, c='cyan', alpha=0.6)
ax.set_title('Padded max_proj + ROI centroids\n(should land ON visible cells, not offset)')
plt.show()


Ly, Lx = 760, 760
raw max_proj shape: (744, 750)
padded fullframe shape: (760, 760)
n_cells: 1197


## Function 1.2 — `click_boundary_points`

Generalizes `estimate_fov_rotation`'s 2-point click into an arbitrary number
of points along the L4 band center, so a curve can be fit through them
instead of a single line slope. Shows the padded max-projection with ROI
centroids overlaid for anatomical context (same convention as
`pick_layer4_peak`'s right panel), since the max-projection alone can be
harder to read than the density-annotated view.

- Left-click to add a point along the L4 band, left-to-right.
- Right-click to remove the last point.
- Press Enter (or middle-click) when done — needs at least 2 points to fit
  a line.

- **Input:** `fullframe_img` (padded max_proj from Function 1),
  `med_coords`, optional `title`.
- **Output:** `points`, an `(n_points, 2)` array of `(x, y)` in pixel
  coordinates, sorted left-to-right by x.

In [37]:
def click_boundary_points(fullframe_img, med_coords,
                           title='Click points along the L4 band center, left to right.\n'
                                 'Right-click = undo last point. Enter = done (need >= 2 points).'):
    """
    Interactively click a series of points tracing the L4 band center across
    the FOV, at as many x-positions as needed to capture how it tilts/curves.

    Parameters
    ----------
    fullframe_img : numpy.ndarray, shape (Ly, Lx)
        Reference image to click on (padded max_proj from Function 1).
    med_coords : numpy.ndarray, shape (n_cells, 2)
        ROI median (y, x) coordinates, overlaid for anatomical context.
    title : str
        Instructions shown as the plot title.

    Returns
    -------
    points : numpy.ndarray, shape (n_points, 2)
        Clicked (x, y) points, sorted left-to-right by x.
    """
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(fullframe_img, cmap='gray',
              vmin=np.percentile(fullframe_img, 2), vmax=np.percentile(fullframe_img, 98))
    ax.scatter(med_coords[:, 1], med_coords[:, 0], s=8, alpha=0.35, c='cyan')
    ax.set_title(title, fontsize=13)
    plt.tight_layout()

    pts = plt.ginput(n=-1, timeout=0)
    plt.close(fig)

    if len(pts) < 2:
        raise ValueError(f"Need at least 2 points to fit a boundary line, got {len(pts)}.")

    points = np.array(sorted(pts, key=lambda p: p[0]))

    print(f"Clicked {len(points)} points (sorted left-to-right):")
    for x, y in points:
        print(f"  x={x:.1f}, y={y:.1f}")

    return points


In [38]:
# --- Try it on the real test session ---
# Click ~4-6 points along the L4 band, left to right, then press Enter.
clicked_points = click_boundary_points(fullframe_maxproj, med_coords_test)


Clicked 4 points (sorted left-to-right):
  x=19.9, y=466.5
  x=185.6, y=413.5
  x=322.3, y=352.8
  x=462.9, y=273.8


## Function 1.3 — `fit_boundary_curve`

Fits a polynomial `y = f(x)` through the clicked points (least-squares via
`np.polyfit`). Default `degree=1` (a tilted line) per your call earlier —
`degree` is exposed as a parameter so we can revisit degree=2 later if a
session clearly shows real curvature, not just tilt.

Guards against the degenerate case where you don't have enough points to
support the requested degree (a line needs >=2 points, a quadratic needs
>=3, etc.) — with only 2-6 points that's an easy mistake to make if degree
is bumped up later without re-clicking more points.

Reports the residuals (clicked y minus fit y at that x) so you can see how
well the line actually matches your clicks — useful now, and more useful
once real curvature shows up and a straight line no longer fits well.

- **Input:** `points` (`(n,2)` array from Function 2), `degree=1`.
- **Output:** `coeffs` (polynomial coefficients, highest power first — as
  `np.polyfit` returns), `curve_fn` (a callable, `curve_fn(x) -> y`, via
  `np.poly1d`).

In [39]:
def fit_boundary_curve(points, degree=1):
    """
    Fit a polynomial y = f(x) through manually-clicked boundary points.

    Parameters
    ----------
    points : numpy.ndarray, shape (n_points, 2)
        (x, y) points from click_boundary_points.
    degree : int
        Polynomial degree. 1 = tilted line (default). Needs at least
        degree+1 points.

    Returns
    -------
    coeffs : numpy.ndarray
        Polynomial coefficients, highest power first (np.polyfit order).
    curve_fn : numpy.poly1d
        Callable: curve_fn(x) -> fitted y.
    """
    n_points = len(points)
    if n_points < degree + 1:
        raise ValueError(
            f"degree={degree} needs at least {degree + 1} points, "
            f"but only {n_points} were given. Re-click more points or lower degree."
        )

    x = points[:, 0]
    y = points[:, 1]

    coeffs = np.polyfit(x, y, degree)
    curve_fn = np.poly1d(coeffs)

    fitted_y = curve_fn(x)
    residuals = y - fitted_y
    rms = np.sqrt(np.mean(residuals ** 2))

    print(f"Fitted degree-{degree} curve through {n_points} points:")
    print(f"  {curve_fn}")
    print(f"  Residuals (clicked y - fit y): min={residuals.min():+.2f}px, "
          f"max={residuals.max():+.2f}px, RMS={rms:.2f}px")

    return coeffs, curve_fn


In [40]:
# --- Fit through your clicked points and eyeball it ---
coeffs, curve_fn = fit_boundary_curve(clicked_points, degree=1)

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(fullframe_maxproj, cmap='gray',
          vmin=np.percentile(fullframe_maxproj, 2), vmax=np.percentile(fullframe_maxproj, 98))
ax.scatter(med_coords_test[:, 1], med_coords_test[:, 0], s=8, alpha=0.25, c='cyan')
ax.scatter(clicked_points[:, 0], clicked_points[:, 1], s=60, c='red', marker='x', label='clicked points')

x_line = np.linspace(0, fullframe_maxproj.shape[1], 200)
ax.plot(x_line, curve_fn(x_line), 'r-', linewidth=2, label=f'fitted degree-1 curve')

ax.set_xlim(0, fullframe_maxproj.shape[1])
ax.set_ylim(fullframe_maxproj.shape[0], 0)
ax.legend(loc='upper right')
ax.set_title('Fitted L4 boundary curve over the FOV\n(does the red line track the clicked points reasonably?)')
plt.tight_layout()
plt.show()


Fitted degree-1 curve through 4 points:
   
-0.4334 x + 484
  Residuals (clicked y - fit y): min=-9.52px, max=+9.91px, RMS=9.23px


## Function 1.4 — `compute_relative_depth`

The curve-based equivalent of the old `depth_y` in `identify_layers`. Instead
of a cell's raw y-coordinate (or its rotated-frame projection), this is how
far *above/below the fitted curve* the cell sits, evaluated at that cell's
own x — so a curve that isn't flat still gives every cell a fair depth
relative to its local L4 position.

- **Input:** `med_coords` (`(n_cells, 2)`, `(y, x)` — same convention as
  everywhere else in this pipeline), `curve_fn` (from Function 3).
- **Output:** `depth_rel`, `(n_cells,)` array. Negative = above the curve
  (toward L2/3), positive = below (toward L5/L6).

## Function 1.5 — `identify_layers_from_curve`

Same layer-bucketing rule as `identify_layers` (±70µm around L4, +150µm for
L5, everything beyond into L6) but thresholding `depth_rel` instead of a
constant `peak_density_y`. Returns the boundary thresholds as **offsets from
the curve** (in pixels) rather than absolute y-values, since "the boundary"
is now a curve, not a single number — Function 6 will need these offsets to
draw the L4-L5 and L5-L6 boundaries as their own curves later.

- **Input:** `med_coords`, `curve_fn`, `um_per_pixel`, `layer4_half_width_um=70`,
  `layer5_offset_um=150`.
- **Output:** `layer_cells` (dict, `{layer_name: cell indices}` — same shape
  as `identify_layers`'s output, so it drops into `MergeTrackedLayerSMI.py`-
  style downstream code unchanged), `boundary_offsets` (dict of pixel
  offsets from the curve: `L4_upper`, `L4_lower`, `L5_lower`).

In [41]:
def compute_relative_depth(med_coords, curve_fn):
    """
    Vertical offset of each cell from the fitted boundary curve, evaluated
    at that cell's own x-position.

    Parameters
    ----------
    med_coords : numpy.ndarray, shape (n_cells, 2)
        (y, x) per cell.
    curve_fn : callable
        curve_fn(x) -> y, from fit_boundary_curve.

    Returns
    -------
    depth_rel : numpy.ndarray, shape (n_cells,)
        y - curve_fn(x) per cell. Negative = above the curve (toward L2/3),
        positive = below (toward L5/L6).
    """
    y = med_coords[:, 0]
    x = med_coords[:, 1]
    return y - curve_fn(x)


def identify_layers_from_curve(med_coords, curve_fn, um_per_pixel,
                                layer4_half_width_um=70, layer5_offset_um=150):
    """
    Assign each cell to a cortical layer based on its vertical offset from a
    fitted boundary curve (Function 4), using the same anatomical thicknesses
    as identify_layers (±70um L4 half-width, 150um L5 offset) but applied
    per-x rather than as a flat cutoff.

    Parameters
    ----------
    med_coords : numpy.ndarray, shape (n_cells, 2)
        (y, x) per cell.
    curve_fn : callable
        Fitted L4-center curve from fit_boundary_curve.
    um_per_pixel : float
        Microns per pixel for this session (e.g. 1.08952017715202 for the
        V1_prism_DREADD animals).
    layer4_half_width_um : float
        Half-width of L4 around the curve, in microns.
    layer5_offset_um : float
        Thickness of L5 below L4, in microns.

    Returns
    -------
    layer_cells : dict
        {layer_name: numpy.ndarray of cell indices} — same shape as
        SpatialModulationIndexLayerSpecific.identify_layers's output.
    boundary_offsets : dict
        {'L4_upper': ..., 'L4_lower': ..., 'L5_lower': ...} in pixels,
        offsets from curve_fn(x). L2/3 is everything above L4_upper, L6 is
        everything below L5_lower.
    """
    depth_rel = compute_relative_depth(med_coords, curve_fn)

    layer4_half_width_px = layer4_half_width_um / um_per_pixel
    layer5_offset_px = layer5_offset_um / um_per_pixel

    l4_upper = -layer4_half_width_px
    l4_lower = layer4_half_width_px
    l5_lower = l4_lower + layer5_offset_px

    layer23_cells = np.where(depth_rel < l4_upper)[0]
    layer4_cells = np.where((depth_rel >= l4_upper) & (depth_rel < l4_lower))[0]
    layer5_cells = np.where((depth_rel >= l4_lower) & (depth_rel < l5_lower))[0]
    layer6_cells = np.where(depth_rel >= l5_lower)[0]

    print(f"L2/3: {len(layer23_cells)} cells (depth_rel < {l4_upper:.1f}px)")
    print(f"L4:   {len(layer4_cells)} cells ({l4_upper:.1f} <= depth_rel < {l4_lower:.1f}px)")
    print(f"L5:   {len(layer5_cells)} cells ({l4_lower:.1f} <= depth_rel < {l5_lower:.1f}px)")
    print(f"L6:   {len(layer6_cells)} cells (depth_rel >= {l5_lower:.1f}px)")

    layer_cells = {
        'L2/3': layer23_cells,
        'L4': layer4_cells,
        'L5': layer5_cells,
        'L6': layer6_cells,
    }
    boundary_offsets = {
        'L4_upper': l4_upper,
        'L4_lower': l4_lower,
        'L5_lower': l5_lower,
    }
    return layer_cells, boundary_offsets


In [42]:
# --- Assign layers from the fitted curve ---
layer_cells_test, boundary_offsets_test = identify_layers_from_curve(
    med_coords_test, curve_fn, um_per_pixel=UM_PER_PIXEL
)
print("\nboundary_offsets:", boundary_offsets_test)
print("Total cells classified:", sum(len(v) for v in layer_cells_test.values()), "/", len(med_coords_test))


L2/3: 209 cells (depth_rel < -64.2px)
L4:   399 cells (-64.2 <= depth_rel < 64.2px)
L5:   287 cells (64.2 <= depth_rel < 201.9px)
L6:   302 cells (depth_rel >= 201.9px)

boundary_offsets: {'L4_upper': -64.24846594670541, 'L4_lower': 64.24846594670541, 'L5_lower': 201.923750118217}
Total cells classified: 1197 / 1197


## Function 1.6 — `review_layer_assignment_popup`

The visual check you asked for before trusting any layer assignment: a
two-panel popup —
- **left:** plain FOV (max-projection), no overlay, so you can look at the
  actual tissue/band structure fresh
- **right:** the same FOV with every ROI colored by its assigned layer, plus
  the fitted L4-band curve and its derived L2/3-L4, L4-L5, L5-L6 boundary
  curves drawn on top, and the original clicked points marked

Ends with a `y/n` prompt (same pattern as the existing rotation-confirm loop
in `SMI_FullSession_Interactive.py`) — `y` accepts, `n` rejects so the caller
knows to let you re-click.

- **Input:** `fullframe_img`, `med_coords`, `layer_cells`, `curve_fn`,
  `boundary_offsets`, `points` (the original clicks, for reference).
- **Output:** `accepted` (bool).

## Function 1.7 — `pick_and_confirm_layers`

The redo loop: click -> fit -> assign -> review, and if you reject it,
discards those points and lets you click again from scratch (not
"adjust" the old points — a fresh, unbiased set is safer than nudging a bad
pick). This is the one function you'll actually call per session going
forward; it wraps Functions 2, 3, 5, 6.

- **Input:** `fullframe_img`, `med_coords`, `um_per_pixel`, `degree=1`,
  `max_attempts=5` (safety cap so a mistaken infinite "n" loop can't hang
  the notebook forever).
- **Output:** `points`, `coeffs`, `curve_fn`, `layer_cells`,
  `boundary_offsets` — everything from the accepted attempt.

In [43]:
_LAYER_COLORS = {
    'L2/3': '#1f77b4',  # Blue  -- matches SpatialModulationIndexLayerSpecific's convention
    'L4': '#ff7f0e',    # Orange
    'L5': '#2ca02c',    # Green
    'L6': '#d62728',    # Red
}


def review_layer_assignment_popup(fullframe_img, med_coords, layer_cells,
                                   curve_fn, boundary_offsets, points):
    """
    Two-panel popup for visually reviewing a layer assignment before
    accepting it: plain FOV on the left, FOV + layer-colored ROIs + boundary
    curves on the right. Press 'y' to accept, 'n' to reject.

    Waits for that keypress using fig.canvas.start_event_loop() in a small
    polling loop -- the same primitive plt.ginput()/BlockingInput use
    internally -- instead of a blocking plt.show(). A bare blocking
    plt.show() was showing a stale/previous frame under Jupyter's own Qt
    event-loop integration (the window that reappeared was actually the
    click-selection figure, not a rendering bug in this figure itself).
    start_event_loop is the officially supported way to wait for a GUI
    event without fighting that integration, which is exactly why
    click_boundary_points's ginput() call has never had this problem.

    Parameters
    ----------
    fullframe_img : numpy.ndarray, shape (Ly, Lx)
        Padded max_proj (or any full-frame reference image).
    med_coords : numpy.ndarray, shape (n_cells, 2)
        (y, x) per cell.
    layer_cells : dict
        {layer_name: cell indices}, from identify_layers_from_curve.
    curve_fn : callable
        Fitted L4-center curve.
    boundary_offsets : dict
        {'L4_upper', 'L4_lower', 'L5_lower'} pixel offsets from curve_fn(x).
    points : numpy.ndarray, shape (n_points, 2)
        The original clicked points, shown for reference.

    Returns
    -------
    accepted : bool
        True if you pressed 'y', False if 'n' (or if closed without either).
    """
    Ly, Lx = fullframe_img.shape
    vmin, vmax = np.percentile(fullframe_img, 2), np.percentile(fullframe_img, 98)

    fig, (ax_plain, ax_layers) = plt.subplots(1, 2, figsize=(16, 8))
    try:
        fig.canvas.manager.set_window_title('LAYER REVIEW -- press y/n')
    except Exception:
        pass

    # --- Left: plain FOV ---
    ax_plain.imshow(fullframe_img, cmap='gray', vmin=vmin, vmax=vmax)
    ax_plain.set_title('FOV only (max-projection)')
    ax_plain.axis('off')

    # --- Right: FOV + layer-colored ROIs + boundary curves ---
    ax_layers.imshow(fullframe_img, cmap='gray', vmin=vmin, vmax=vmax)

    for layer_name, cell_idx in layer_cells.items():
        if len(cell_idx) == 0:
            continue
        color = _LAYER_COLORS.get(layer_name, 'white')
        ax_layers.scatter(med_coords[cell_idx, 1], med_coords[cell_idx, 0],
                           s=14, alpha=0.85, color=color,
                           label=f'{layer_name} ({len(cell_idx)})')

    x_line = np.linspace(0, Lx, 200)
    ax_layers.plot(x_line, curve_fn(x_line), '--', color='yellow', linewidth=1.5,
                    label='L4 center (clicked)')
    ax_layers.plot(x_line, curve_fn(x_line) + boundary_offsets['L4_upper'], '--',
                    color='white', linewidth=1.2, label='L2/3-L4')
    ax_layers.plot(x_line, curve_fn(x_line) + boundary_offsets['L4_lower'], '--',
                    color='white', linewidth=1.2, label='L4-L5')
    ax_layers.plot(x_line, curve_fn(x_line) + boundary_offsets['L5_lower'], '--',
                    color='white', linewidth=1.2, label='L5-L6')
    ax_layers.scatter(points[:, 0], points[:, 1], s=60, c='red', marker='x',
                       label='clicked points', zorder=10)

    ax_layers.set_xlim(0, Lx)
    ax_layers.set_ylim(Ly, 0)
    ax_layers.set_title('FOV + layer assignment')
    ax_layers.axis('off')
    ax_layers.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)

    decision = {'accepted': None}

    def on_key(event):
        if event.key == 'y':
            decision['accepted'] = True
            fig.canvas.stop_event_loop()
        elif event.key == 'n':
            decision['accepted'] = False
            fig.canvas.stop_event_loop()

    fig.canvas.mpl_connect('key_press_event', on_key)
    fig.suptitle("Click this figure to focus it, then press 'y' to ACCEPT or 'n' to REJECT",
                 fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.show(block=False)

    while decision['accepted'] is None and plt.fignum_exists(fig.number):
        fig.canvas.start_event_loop(0.1)

    if plt.fignum_exists(fig.number):
        plt.close(fig)

    if decision['accepted'] is None:
        print("Window closed without pressing y/n -- treating as reject.")
        return False

    return decision['accepted']


In [44]:
def pick_and_confirm_layers(fullframe_img, med_coords, um_per_pixel, degree=1, max_attempts=5):
    """
    Click -> fit -> assign -> review, looping until you accept the result
    (or max_attempts is hit). Each rejected attempt discards its points
    entirely -- you re-click from scratch next round.

    Parameters
    ----------
    fullframe_img : numpy.ndarray, shape (Ly, Lx)
        Padded max_proj reference image.
    med_coords : numpy.ndarray, shape (n_cells, 2)
        (y, x) per cell.
    um_per_pixel : float
        Microns per pixel for this session.
    degree : int
        Polynomial degree for the boundary fit (default 1).
    max_attempts : int
        Safety cap on re-click rounds.

    Returns
    -------
    points, coeffs, curve_fn, layer_cells, boundary_offsets
        From the accepted attempt.
    """
    for attempt in range(1, max_attempts + 1):
        print(f"\n{'='*60}\nAttempt {attempt}/{max_attempts}\n{'='*60}")

        points = click_boundary_points(fullframe_img, med_coords)
        coeffs, curve_fn = fit_boundary_curve(points, degree=degree)
        layer_cells, boundary_offsets = identify_layers_from_curve(
            med_coords, curve_fn, um_per_pixel=um_per_pixel
        )

        accepted = review_layer_assignment_popup(
            fullframe_img, med_coords, layer_cells, curve_fn, boundary_offsets, points
        )

        if accepted:
            print(f"Accepted on attempt {attempt}.")
            return points, coeffs, curve_fn, layer_cells, boundary_offsets

        print("Rejected -- re-click from scratch.")

    raise RuntimeError(f"No accepted layer assignment after {max_attempts} attempts.")


In [45]:
# --- Full end-to-end test: click, fit, assign, review, (re-click if you reject it) ---
(final_points, final_coeffs, final_curve_fn,
 final_layer_cells, final_boundary_offsets) = pick_and_confirm_layers(
    fullframe_maxproj, med_coords_test, um_per_pixel=UM_PER_PIXEL
)



Attempt 1/5
Clicked 5 points (sorted left-to-right):
  x=20.9, y=475.1
  x=186.5, y=409.6
  x=302.1, y=339.3
  x=386.9, y=296.0
  x=514.9, y=221.8
Fitted degree-1 curve through 5 points:
   
-0.5183 x + 494.7
  Residuals (clicked y - fit y): min=-8.73px, max=+11.63px, RMS=7.10px
L2/3: 178 cells (depth_rel < -64.2px)
L4:   364 cells (-64.2 <= depth_rel < 64.2px)
L5:   322 cells (64.2 <= depth_rel < 201.9px)
L6:   333 cells (depth_rel >= 201.9px)
Rejected -- re-click from scratch.

Attempt 2/5
Clicked 5 points (sorted left-to-right):
  x=19.0, y=470.3
  x=190.4, y=408.7
  x=302.1, y=336.4
  x=454.3, y=249.8
  x=532.3, y=226.7
Fitted degree-1 curve through 5 points:
   
-0.5002 x + 488.2
  Residuals (clicked y - fit y): min=-11.24px, max=+15.68px, RMS=9.65px
L2/3: 172 cells (depth_rel < -64.2px)
L4:   374 cells (-64.2 <= depth_rel < 64.2px)
L5:   317 cells (64.2 <= depth_rel < 201.9px)
L6:   334 cells (depth_rel >= 201.9px)
Accepted on attempt 2.


## Function 1.8 — `layer_cells_to_labels`

Converts the `layer_cells` dict (`{layer_name: indices}`) into a flat
per-cell integer label array, aligned 1:1 with `med_coords`/`cell_idx` order.
This is the form that's actually easy to persist and join later — a dict of
index arrays doesn't round-trip cleanly through HDF5, a flat labeled array
does.

- **Input:** `layer_cells`, `n_cells`, `layer_names` (fixes the code order,
  default `('L2/3', 'L4', 'L5', 'L6')`).
- **Output:** `layer_codes` (`(n_cells,)` int8 array, `-1` for any
  unclassified cell — shouldn't happen given `identify_layers_from_curve`
  is exhaustive, but guarded rather than assumed), `layer_names` (echoed
  back, so the save step doesn't need it passed separately).

## Function 1.9 — `save_layer_curve_results` / `load_layer_curve_results`

Persists one session's Phase 1 output as `{session_label}_layer_curve_results.h5`,
saved **directly in the session's TSeries folder** — the same location
`preproc.h5` and (eventually) `*_smi_results.h5` live, discoverable the same
way (`glob.glob(session_dir, "*_layer_curve_results.h5")`).

Contents: `cell_idx` (the iscell-filtered ROI indices — a guard so we can
verify this file's cell ordering still matches a session's current
`stat.npy`/`iscell.npy` if suite2p is ever re-run), `layer_codes` +
`layer_names` (the per-cell labels), `points`, `coeffs`, `degree`,
`um_per_pixel`, and `boundary_offsets` (so the exact curve can be
reconstructed later for plotting/QC without re-fitting).

- **Input (save):** `save_path`, `session_label`, `cell_idx`, `layer_codes`,
  `layer_names`, `points`, `coeffs`, `degree`, `um_per_pixel`,
  `boundary_offsets`.
- **Input (load):** `save_path`.
- **Output (load):** dict with all of the above, plus `layer_of_cell` — a
  `{roi_idx: layer_name}` dict in the exact shape
  `MergeTrackedLayerSMI.load_smi_results` already expects, so a future
  Phase-2 join can use either file interchangeably.

In [46]:
import h5py


def layer_cells_to_labels(layer_cells, n_cells, layer_names=('L2/3', 'L4', 'L5', 'L6')):
    """
    Flatten a {layer_name: indices} dict into a per-cell integer label array.

    Parameters
    ----------
    layer_cells : dict
        {layer_name: numpy.ndarray of cell indices}, from identify_layers_from_curve.
    n_cells : int
        Total number of cells (length of the output array).
    layer_names : tuple of str
        Fixes the code order: layer_names[code] -> name. Must cover every
        key actually present in layer_cells.

    Returns
    -------
    layer_codes : numpy.ndarray, shape (n_cells,), dtype int8
        Code per cell; -1 if a cell wasn't assigned to any layer.
    layer_names : tuple of str
        Echoed back for convenience.
    """
    layer_codes = np.full(n_cells, -1, dtype=np.int8)
    for code, name in enumerate(layer_names):
        if name in layer_cells:
            layer_codes[layer_cells[name]] = code

    n_unassigned = np.sum(layer_codes == -1)
    if n_unassigned > 0:
        print(f"WARNING: {n_unassigned}/{n_cells} cells have no layer code "
              f"(layer_cells covered names other than {layer_names}?).")

    return layer_codes, layer_names


def save_layer_curve_results(save_path, session_label, cell_idx, layer_codes, layer_names,
                              points, coeffs, degree, um_per_pixel, boundary_offsets):
    """
    Save one session's Phase 1 output to HDF5.

    Parameters
    ----------
    save_path : str
        Full path to write, e.g. os.path.join(session_dir, f'{session_label}_layer_curve_results.h5').
    session_label : str
    cell_idx : numpy.ndarray
        iscell-filtered ROI indices (session_data['cell_idx'] convention).
    layer_codes : numpy.ndarray
    layer_names : tuple of str
    points : numpy.ndarray, shape (n_points, 2)
    coeffs : numpy.ndarray
    degree : int
    um_per_pixel : float
    boundary_offsets : dict
        {'L4_upper', 'L4_lower', 'L5_lower'}.
    """
    with h5py.File(save_path, 'w') as f:
        f.attrs['session_label'] = session_label
        f.attrs['degree'] = degree
        f.attrs['um_per_pixel'] = um_per_pixel
        for key, val in boundary_offsets.items():
            f.attrs[f'boundary_offset_{key}'] = val

        f.create_dataset('cell_idx', data=cell_idx)
        f.create_dataset('layer_codes', data=layer_codes)
        f.create_dataset('layer_names', data=np.array(layer_names, dtype='S10'))
        f.create_dataset('points', data=points)
        f.create_dataset('coeffs', data=coeffs)

    print(f"Saved layer curve results for '{session_label}' -> {save_path}")


def load_layer_curve_results(save_path):
    """
    Load one session's Phase 1 output back from HDF5.

    Returns
    -------
    result : dict with keys:
        session_label, degree, um_per_pixel, boundary_offsets, cell_idx,
        layer_codes, layer_names, points, coeffs, curve_fn, layer_of_cell
        ({roi_idx: layer_name}, MergeTrackedLayerSMI-compatible shape).
    """
    with h5py.File(save_path, 'r') as f:
        session_label = f.attrs['session_label']
        degree = int(f.attrs['degree'])
        um_per_pixel = float(f.attrs['um_per_pixel'])
        boundary_offsets = {
            'L4_upper': float(f.attrs['boundary_offset_L4_upper']),
            'L4_lower': float(f.attrs['boundary_offset_L4_lower']),
            'L5_lower': float(f.attrs['boundary_offset_L5_lower']),
        }
        cell_idx = f['cell_idx'][:]
        layer_codes = f['layer_codes'][:]
        layer_names = tuple(n.decode() if isinstance(n, bytes) else n
                             for n in f['layer_names'][:])
        points = f['points'][:]
        coeffs = f['coeffs'][:]

    curve_fn = np.poly1d(coeffs)
    layer_of_cell = {
        int(roi_idx): layer_names[code]
        for roi_idx, code in zip(cell_idx, layer_codes) if code >= 0
    }

    return {
        'session_label': session_label,
        'degree': degree,
        'um_per_pixel': um_per_pixel,
        'boundary_offsets': boundary_offsets,
        'cell_idx': cell_idx,
        'layer_codes': layer_codes,
        'layer_names': layer_names,
        'points': points,
        'coeffs': coeffs,
        'curve_fn': curve_fn,
        'layer_of_cell': layer_of_cell,
    }


In [47]:
# --- Round-trip test: convert, save, reload, and verify nothing changed ---
layer_codes_test, layer_names_test = layer_cells_to_labels(final_layer_cells, len(med_coords_test))

test_save_path = os.path.join(TEST_SESSION_DIR, 'TEST_layer_curve_results.h5')
save_layer_curve_results(
    test_save_path, session_label=os.path.basename(TEST_SESSION_DIR),
    cell_idx=cell_idx_test, layer_codes=layer_codes_test, layer_names=layer_names_test,
    points=final_points, coeffs=final_coeffs, degree=1,
    um_per_pixel=UM_PER_PIXEL, boundary_offsets=final_boundary_offsets
)

reloaded = load_layer_curve_results(test_save_path)
print("\nRound-trip check:")
print("  layer_codes match:", np.array_equal(reloaded['layer_codes'], layer_codes_test))
print("  cell_idx match:", np.array_equal(reloaded['cell_idx'], cell_idx_test))
print("  points match:", np.allclose(reloaded['points'], final_points))
print("  n cells in layer_of_cell:", len(reloaded['layer_of_cell']), "/", len(med_coords_test))
print("  example entries:", dict(list(reloaded['layer_of_cell'].items())[:5]))

os.remove(test_save_path)  # this was just a round-trip check, not a real Phase 1 output yet
print("\n(Removed test file -- process_session_layer_curve will do the real save later.)")


Saved layer curve results for 'TSeries-07232026-0843-001' -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260723_JSY_JSY093_LongitudinalImaging_DREADD_Day5\TSeries-07232026-0843-001\TEST_layer_curve_results.h5

Round-trip check:
  layer_codes match: True
  cell_idx match: True
  points match: True
  n cells in layer_of_cell: 1197 / 1197
  example entries: {0: 'L6', 1: 'L2/3', 2: 'L5', 3: 'L5', 4: 'L5'}

(Removed test file -- process_session_layer_curve will do the real save later.)


> **Known open issue:** the review popup still sometimes needs a rejected
> attempt before it renders/responds correctly (see conversation). Workable
> for now while we finish building, but worth revisiting before running this
> across the full ~13-session-per-animal batch — re-clicking twice every
> single session would be tedious. Come back to this before Phase 1 is
> actually used for real data collection, not just development.

## Function 1.10 — `plot_curve_boundary_consistency`

Cross-session QC: overlays every session's fitted L4 curve on one reference
FOV (translation-aligned via `phase_cross_correlation`, same registration
step `TrackROIs_SalineDCZ.py`/`LayerBoundaryParams.py` already use), so
drift or an outlier click session is visible before trusting any of them
for layer-resolved analysis. Direct curve-based counterpart to
`LayerBoundaryParams.plot_boundary_consistency`.

- **Input:** `session_labels`, `fullframe_imgs` (parallel lists),
  `curve_results` (`{session_label: load_layer_curve_results(...) dict}`),
  `reference_label` (defaults to the first available), `save_path`.
- **Output:** `fig`.

In [48]:
from skimage.registration import phase_cross_correlation


def plot_curve_boundary_consistency(session_labels, fullframe_imgs, curve_results,
                                     reference_label=None, save_path=None):
    """
    Overlay each session's fitted L4 curve on one reference FOV, aligned by
    translation only (phase_cross_correlation), so drift/outlier picks are
    visible across a longitudinal series before trusting them.

    Parameters
    ----------
    session_labels : list of str
    fullframe_imgs : list of numpy.ndarray
        Same order as session_labels (padded max_proj per session).
    curve_results : dict
        {session_label: load_layer_curve_results(...) dict} -- needs at
        least 'curve_fn' and 'degree' per entry.
    reference_label : str, optional
        Defaults to the first session_label with an entry in curve_results.
    save_path : str, optional

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    available = [s for s in session_labels if s in curve_results]
    if len(available) == 0:
        raise ValueError("None of the given session_labels have entries in curve_results")

    if reference_label is None:
        reference_label = available[0]
    if reference_label not in available:
        raise ValueError(f"reference_label '{reference_label}' has no entry in curve_results")

    label_to_img = dict(zip(session_labels, fullframe_imgs))
    ref_img = label_to_img[reference_label]

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(ref_img, cmap='gray',
              vmin=np.percentile(ref_img, 2), vmax=np.percentile(ref_img, 98))

    colors = plt.cm.hsv(np.linspace(0, 0.85, len(available)))

    for color, label in zip(colors, available):
        img = label_to_img[label]
        curve_fn = curve_results[label]['curve_fn']

        if label == reference_label:
            dy, dx = 0.0, 0.0
        else:
            shift_yx, _, _ = phase_cross_correlation(ref_img, img, upsample_factor=10)
            dy, dx = shift_yx[0], shift_yx[1]

        x_line = np.linspace(0, img.shape[1], 200)
        y_line = curve_fn(x_line)
        ax.plot(x_line + dx, y_line + dy, '-', color=color, linewidth=2,
                 label=f"{label} (deg={curve_results[label]['degree']})")

    ax.set_title(f"L4 boundary curve consistency across sessions\n(reference: {reference_label})")
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
    ax.axis('off')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Boundary consistency figure saved to {save_path}")

    return fig


## Function 1.11 — `process_session_layer_curve`

The real per-session entry point: load a session's suite2p output, build the
padded max-projection + `med_coords`/`cell_idx`, run the full
click→fit→assign→review loop, then save the result. Wraps Functions 1, 7-9.

`session_label` defaults to the TSeries folder's own basename rather than
its parent Day-folder (the convention `SMI_FullSession_Interactive.py` uses)
-- for the flat `TSeries-..._SALINE-001` / `TSeries-..._DCZ-001` layout,
SALINE and DCZ share the same parent folder, so a parent-folder label would
collide between the two conditions.

- **Input:** `session_dir`, `um_per_pixel`, `session_label=None`,
  `degree=1`, `max_attempts=5`.
- **Output:** dict with `session_label`, `save_path`, `fullframe_img`,
  `med_coords`, `cell_idx`, `points`, `coeffs`, `curve_fn`, `layer_cells`,
  `boundary_offsets`.

## Function 1.12 — `process_animal_layer_curve`

Loops Function 11 over every session for one animal, then produces the
cross-session consistency plot (Function 10). Mirrors
`SMI_FullSession_Interactive.process_animal`'s structure/config shape
(`animal_id`, `store_dir`, `um_per_pixel`, `sessions`) so it's a familiar
shape if you've used that script before.

- **Input:** `animal_cfg` dict — `animal_id`, `store_dir`, `sessions`
  (list of session_dir paths), `um_per_pixel`, `degree=1`.
- **Output:** `session_labels`, `fullframe_imgs`, `curve_results` (the same
  three things `plot_curve_boundary_consistency` needs, handed back in case
  you want to re-plot or inspect further).

In [49]:
def process_session_layer_curve(session_dir, um_per_pixel, session_label=None,
                                 degree=1, max_attempts=5):
    """
    End-to-end Phase 1 for one session: load -> click/fit/assign/review loop
    -> save. See module docstring above for parameter/return details.
    """
    if session_label is None:
        session_label = os.path.basename(session_dir)

    plane0_path = os.path.join(session_dir, 'suite2p', 'plane0')
    ops = np.load(os.path.join(plane0_path, 'ops.npy'), allow_pickle=True).item()
    stat = np.load(os.path.join(plane0_path, 'stat.npy'), allow_pickle=True)
    iscell = np.load(os.path.join(plane0_path, 'iscell.npy'), allow_pickle=True)

    cell_idx = np.where(iscell[:, 0] == 1)[0]
    med_coords = np.array([stat[i]['med'] for i in cell_idx])
    fullframe_img = get_fullframe_max_proj(ops)

    print(f"\n{session_label}: {len(cell_idx)} cells")

    points, coeffs, curve_fn, layer_cells, boundary_offsets = pick_and_confirm_layers(
        fullframe_img, med_coords, um_per_pixel=um_per_pixel,
        degree=degree, max_attempts=max_attempts
    )

    layer_codes, layer_names = layer_cells_to_labels(layer_cells, len(med_coords))

    save_path = os.path.join(session_dir, f'{session_label}_layer_curve_results.h5')
    save_layer_curve_results(
        save_path, session_label, cell_idx, layer_codes, layer_names,
        points, coeffs, degree, um_per_pixel, boundary_offsets
    )

    return {
        'session_label': session_label,
        'save_path': save_path,
        'fullframe_img': fullframe_img,
        'med_coords': med_coords,
        'cell_idx': cell_idx,
        'points': points,
        'coeffs': coeffs,
        'curve_fn': curve_fn,
        'layer_cells': layer_cells,
        'boundary_offsets': boundary_offsets,
    }


def process_animal_layer_curve(animal_cfg):
    """
    Loop process_session_layer_curve over one animal's sessions, then plot
    cross-session boundary consistency.

    Parameters
    ----------
    animal_cfg : dict
        Keys: 'animal_id', 'store_dir', 'sessions' (list of session_dir
        paths), 'um_per_pixel', 'degree' (optional, default 1).

    Returns
    -------
    session_labels, fullframe_imgs, curve_results
    """
    animal_id = animal_cfg['animal_id']
    store_dir = animal_cfg['store_dir']
    um_per_pixel = animal_cfg['um_per_pixel']
    degree = animal_cfg.get('degree', 1)
    sessions = animal_cfg['sessions']

    print("\n" + "=" * 90)
    print(f" PHASE 1 LAYER ASSIGNMENT: {animal_id}")
    print("=" * 90)

    session_labels = []
    fullframe_imgs = []
    curve_results = {}

    for session_dir in sessions:
        if not os.path.isdir(session_dir):
            print(f"\nSkipped (folder not found): {session_dir}")
            continue

        result = process_session_layer_curve(
            session_dir, um_per_pixel=um_per_pixel, degree=degree
        )
        session_labels.append(result['session_label'])
        fullframe_imgs.append(result['fullframe_img'])
        curve_results[result['session_label']] = load_layer_curve_results(result['save_path'])

    if len(session_labels) >= 2:
        fig_path = os.path.join(store_dir, f'{animal_id}_curve_boundary_consistency.png')
        plot_curve_boundary_consistency(session_labels, fullframe_imgs, curve_results,
                                         save_path=fig_path)
    else:
        print("\nFewer than 2 sessions processed -- skipping consistency plot.")

    return session_labels, fullframe_imgs, curve_results


In [ ]:
# --- Real-use template ---
# Fill in the actual session list for an animal and call process_animal_layer_curve.
# One entry per session, in whatever order you want the consistency plot legend to show.

ANIMALS_LAYER_CURVE = [
    {
        'animal_id': 'JSY090',
        'store_dir': r'D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD',
        'um_per_pixel': 1.08952017715202,
        'degree': 1,
        'sessions': [
            r'D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260722_JSY_JSY090_LongitudinalImaging_DREADD_Day4\TSeries-07222026-1831-002',
            r'D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260723_JSY_JSY090_LongitudinalImaging_DREADD_Day5\TSeries-07232026-0843-001'
        ],
    },
    # {
    #     'animal_id': 'JSY093',
    #     'store_dir': r'D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD',
    #     'um_per_pixel': 1.08952017715202,
    #     'degree': 1,
    #     'sessions': [...],
    # },
]

# Keep each animal's (session_labels, fullframe_imgs, curve_results) around --
# the diagnostics cell below needs them, not just the saved consistency figure.
animal_results = {}
for animal_cfg in ANIMALS_LAYER_CURVE:
    animal_results[animal_cfg['animal_id']] = process_animal_layer_curve(animal_cfg)


## Function 1.13 — `summarize_layer_curve_consistency`

The numeric complement to the consistency plot: a per-session table of the
fitted line's slope/intercept and each layer's cell count/percentage, so
"do these two sessions roughly agree" is a comparison of numbers, not just
eyeballing an overlay of curves. A big jump in, say, L4's percentage between
Day4 and Day5 for the *same animal* (no reason to expect real layer-size
change day to day) would flag an inconsistent click before it propagates
into Phase 2/3.

- **Input:** `session_labels`, `curve_results` (both returned by
  `process_animal_layer_curve`).
- **Output:** `df` (pandas DataFrame, one row per session), also printed.

In [ ]:
import pandas as pd


def summarize_layer_curve_consistency(session_labels, curve_results):
    """
    Build a per-session summary table: fitted curve slope/intercept plus
    each layer's cell count/percentage, for numerically comparing sessions
    that a consistency plot can only show visually.

    Parameters
    ----------
    session_labels : list of str
    curve_results : dict
        {session_label: load_layer_curve_results(...) dict}.

    Returns
    -------
    df : pandas.DataFrame
        One row per session in session_labels.
    """
    rows = []
    for label in session_labels:
        r = curve_results[label]
        layer_codes = r['layer_codes']
        layer_names = r['layer_names']
        n_cells = len(layer_codes)

        row = {
            'session_label': label,
            'n_cells': n_cells,
            'degree': r['degree'],
            'slope': r['coeffs'][0] if r['degree'] == 1 else np.nan,
            'intercept': r['coeffs'][-1],
            'n_clicked_points': len(r['points']),
        }
        for code, name in enumerate(layer_names):
            count = int(np.sum(layer_codes == code))
            row[f'{name}_n'] = count
            row[f'{name}_pct'] = 100 * count / n_cells if n_cells else np.nan
        rows.append(row)

    df = pd.DataFrame(rows)
    with pd.option_context('display.width', 140, 'display.max_columns', None):
        print(df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

    return df


In [ ]:
# --- Multi-session diagnostic on JSY090's Day4/Day5 pair ---
jsy090_session_labels, jsy090_fullframe_imgs, jsy090_curve_results = animal_results['JSY090']

print(f"Sessions processed: {jsy090_session_labels}\n")
summary_df = summarize_layer_curve_consistency(jsy090_session_labels, jsy090_curve_results)

# Re-display the consistency plot inline (already saved to store_dir by
# process_animal_layer_curve, this just shows it in the notebook too)
plot_curve_boundary_consistency(jsy090_session_labels, jsy090_fullframe_imgs, jsy090_curve_results)
plt.show()
